In [0]:
# src/exploration/eda_bronze.py
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, when

# -------- CONFIG (adapte tes paths) --------
# Exemple: fichiers CSV déjà extraits
# RAW_GLOB = "dbfs:/FileStore/bls_cew/extracted_v/*.csv"
# ou si tu utilises Volume:
RAW_GLOB = "/Volumes/bls_cew/raw/bronze_cew/*.csv"

# -------- READ --------
df_raw = spark.read.csv(RAW_GLOB, header=True, inferSchema=True)

print("=== SCHEMA ===")
df_raw.printSchema()

total_rows = df_raw.count()
print(f"Nombre de lignes brutes : {total_rows}")
print(f"Nombre de colonnes : {len(df_raw.columns)}")
print("Colonnes :", df_raw.columns)

# -------- SAMPLE --------
display(df_raw.limit(5))

# -------- NULL COUNTS (only columns having nulls) --------
null_counts = df_raw.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
])

null_counts_long = null_counts.selectExpr(
    "stack({0}, {1}) as (column_name, null_count)".format(
        len(df_raw.columns),
        ", ".join([f"'{c}', {c}" for c in df_raw.columns])
    )
)

null_columns_only = null_counts_long.filter(col("null_count") > 0).orderBy(F.desc("null_count"))

print("=== Columns with NULLs ===")
null_columns_only.show(200, truncate=False)

# -------- DUPLICATES (whole row) --------
dup_count = df_raw.count() - df_raw.dropDuplicates().count()
print("Doublons (ligne complète) :", dup_count)

# -------- NEGATIVE CHECK example --------
if "annual_avg_emplvl" in df_raw.columns:
    neg_empl = df_raw.filter(F.col("annual_avg_emplvl") < 0).count()
    print("Valeurs négatives annual_avg_emplvl :", neg_empl)

# -------- QUICK DISTRIBUTION --------
if "year" in df_raw.columns:
    print("=== Rows per year ===")
    df_raw.groupBy("year").count().orderBy("year").show(50, truncate=False)